# HW6 — SQL Practice and Classes

Name: Alex Devoid  
Course: ST 554

In [25]:

import warnings
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import (
    LinearRegression,
    LassoCV,
    RidgeCV,
    ElasticNetCV,
    LogisticRegression,
    LogisticRegressionCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, log_loss, accuracy_score


# Random seed to can reproduce the same split and model results later on.
RANDOM_STATE = 1234


## 1. Read in and Combine Data

I start by reading the red and white wine files separately. Then I stack them into one data frame and add a wine-type variable so the source of each observation is still clear after the combine step.

I also create `type_binary` as a helper 0/1 version for the modeling sections, since scikit-learn needs numeric inputs once wine type enters a model.


In [26]:
# Read the two source files directly from the UCI repository.
red_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
white_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

# Adingd wine type before combining so I can keep both a readable label and a numeric version for sklearn.
red_wine = pd.read_csv(red_url, sep=';').assign(type='red', type_binary=0)
white_wine = pd.read_csv(white_url, sep=';').assign(type='white', type_binary=1)
wine = pd.concat([red_wine, white_wine], ignore_index=True)

# Checks before moving on to the split and modeling sections.
row_summary = pd.DataFrame(
    {
        'rows': [len(red_wine), len(white_wine), len(wine)],
        'columns': [red_wine.shape[1], white_wine.shape[1], wine.shape[1]],
    },
    index=['red_wine', 'white_wine', 'combined'],
)
row_summary.index.name = 'dataset'

type_summary = wine['type'].value_counts().rename_axis('type').to_frame('count')
type_summary['proportion'] = type_summary['count'] / len(wine)
missing_cells = int(wine.isna().sum().sum())

# show row counts, a look at the data, and the type balance.
display(row_summary)
display(wine.head())
display(type_summary)




,rows,columns
dataset,,
red_wine,1599,14
white_wine,4898,14
combined,6497,14


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_binary
0,7.4000,0.7000,0.0000,1.9000,0.0760,11.0000,34.0000,0.9978,3.5100,0.5600,9.4000,5,red,0
1,7.8000,0.8800,0.0000,2.6000,0.0980,25.0000,67.0000,0.9968,3.2000,0.6800,9.8000,5,red,0
2,7.8000,0.7600,0.0400,2.3000,0.0920,15.0000,54.0000,0.9970,3.2600,0.6500,9.8000,5,red,0
3,11.2000,0.2800,0.5600,1.9000,0.0750,17.0000,60.0000,0.9980,3.1600,0.5800,9.8000,6,red,0
4,7.4000,0.7000,0.0000,1.9000,0.0760,11.0000,34.0000,0.9978,3.5100,0.5600,9.4000,5,red,0


,count,proportion
type,,
white,4898,0.7539
red,1599,0.2461


The combined data set has 6,497 rows and 14 columns after adding the wine-type label and a 0/1 encoding. There are 4,898 white wines and 1,599 red wines, so the classes are not perfectly balanced. The data have 0 missing cells, so I do not need a missing-value imputation step before modeling.

## 2. Split the Data


In [27]:

base_physicochemical_features = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'density',
    'pH',
    'sulphates',
]
# Adding the predictor for regression and classification features
regression_features = base_physicochemical_features + ['type_binary']
classification_features = base_physicochemical_features + ['alcohol']

# Stratify on the readable wine-type label so the train/test split preserves class balance.
train_df, test_df = train_test_split(
    wine,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=wine['type'],
)

# helper to compare type counts across the full data, training set, and test set.
def type_share(frame, label):
    counts = frame['type'].value_counts().reindex(['red', 'white'])
    return pd.DataFrame(
        {
            'split': label,
            'type': counts.index,
            'count': counts.values,
            'proportion': counts.values / len(frame),
        }
    )

# Stack the three summaries so I can check the proportions side by side.
split_summary = pd.concat(
    [
        type_share(wine, 'full_data'),
        type_share(train_df, 'training'),
        type_share(test_df, 'test'),
    ],
    ignore_index=True,
)


display(split_summary)




,split,type,count,proportion
0,full_data,red,1599,0.2461
1,full_data,white,4898,0.7539
2,training,red,1199,0.2461
3,training,white,3673,0.7539
4,test,red,400,0.2462
5,test,white,1225,0.7538


The training set has 4,872 observations and the test set has 1,625 observations. The red/white proportions stay very close across the full data, training set, and test set.